In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 19
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.523794098938023

Trial 1 =========================================
14.256085908009771



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 2 =========================================
15.676762932467813



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 3 =========================================
16.208184694662865

Trial 4 =========================================
16.193852929748704

Trial 5 =========================================
14.052506568383409



...

Trial 83 =========================================
15.962107588166912

Trial 84 =========================================
14.101852188555904



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 85 =========================================
14.55443487254233

Trial 86 =========================================
16.033009364014884

Trial 87 =========================================
14.18221736862668

Trial 88 =========================================
13.889985679586928

Trial 89 =========================================
14.29015917575543

Trial 90 =========================================
14.08554698626161

Trial 91 =========================================
17.114985289037147

Trial 92 =========================================
15.816065379068563

Trial 93 =========================================
14.621238903518535

Trial 94 =========================================
14.239326944478176

Trial 95 =========================================
15.329096390179544

Trial 96 =========================================
14.776415667216629



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 97 =========================================
15.727576726755204

Trial 98 =========================================
14.07234403956928

Trial 99 =========================================
15.259561323389704

[15.5237941  14.25608591 15.67676293 16.20818469 16.19385293 14.05250657
 16.20239455 14.23050206 13.77543744 13.81394696 14.14799318 16.20280232
 15.92922201 17.51831728 16.03747099 15.13695152 15.31156717 14.56763008
 13.4293754  14.77973859 18.66179342 15.07542386 14.41186215 15.75021249
 15.3840624  14.42420867 14.08218638 16.17040196 16.05016261 13.78326138
 14.59581663 14.27133299 18.68056972 14.29530394 15.891591   14.07241051
 17.38610011 13.56308818 14.15004409 15.57797613 13.78846982 13.80722628
 13.9731288  15.02310964 14.79351383 15.92922201 14.86516789 15.86059197
 14.24488148 15.27673528 15.2484666  16.51112097 14.72831155 14.22090718
 16.16372396 14.0200583  15.37581527 15.87592551 15.86896282 13.38868658
 15.49351539 14.16226222 14.1061017  14.58657232 16.236348

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.68857497254335
Avg = 15.120382341296688
Std = 1.1374704543357805


In [6]:
print(y_max_arr.tolist())

[15.523794098938023, 14.256085908009771, 15.676762932467813, 16.208184694662865, 16.193852929748704, 14.052506568383409, 16.20239454971047, 14.230502061567536, 13.775437435951462, 13.813946960881514, 14.147993177086141, 16.20280232178335, 15.929222010399386, 17.518317281759405, 16.037470985379837, 15.13695152386112, 15.311567166275882, 14.567630075252023, 13.42937539529881, 14.779738589529174, 18.66179342410212, 15.075423857111158, 14.411862146418034, 15.75021248953206, 15.38406240210761, 14.42420866887363, 14.08218638298148, 16.170401961349313, 16.050162606368424, 13.78326138081844, 14.595816634281446, 14.271332988458214, 18.680569724920797, 14.295303940362466, 15.891590996996257, 14.07241050784095, 17.386100108583584, 13.56308817518874, 14.15004409136208, 15.577976131462913, 13.788469815951009, 13.807226284132256, 13.973128801606961, 15.023109641519598, 14.79351382901924, 15.929222010399386, 14.865167894633112, 15.860591972315097, 14.244881479124603, 15.276735276537938, 15.2484665961

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-19.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)